# Placement Readiness Predictor
## GDG-USAR AIML Technical Task 1

Complete end-to-end classification pipeline: cleaning, feature selection, feature engineering, preprocessing, two-model comparison, required metrics, example prediction, interpretability and model export.

## 1. Imports

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

## 2. Load the dataset

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
except ImportError:
    candidates = ["data/placement_readiness.csv", "data/Placement_Readiness_Dataset.csv"]
    file_name = next((x for x in candidates if os.path.exists(x)), None)
    if file_name is None:
        raise FileNotFoundError("Put the supplied CSV in data/ or run this notebook in Colab.")

df = pd.read_csv(file_name)
print("File:", file_name)
print("Shape:", df.shape)
display(df.head())

## 3. Inspect data

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget:")
display(df["placement"].value_counts(dropna=False))

## 4. Remove duplicates

In [ ]:
before = len(df)
duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print("Rows before:", before)
print("Duplicates removed:", duplicates)
print("Rows after:", len(df))

## 5. Target encoding

In [ ]:
TARGET = "placement"
y = df[TARGET].map({"Not Placed": 0, "Placed": 1})
if y.isna().any():
    raise ValueError("Unexpected target values: " + str(df.loc[y.isna(), TARGET].unique()))
display(y.value_counts().rename(index={0:"Not Placed",1:"Placed"}))

## 6. Feature selection

In [ ]:
drop_columns = ["name", "email", "other_course", "opinion", TARGET]
X = df.drop(columns=drop_columns).copy()
print(X.columns.tolist())

## 7. Feature engineering

In [ ]:
X["languages"] = X["languages"].fillna("").astype(str)
X["language_count"] = X["languages"].apply(lambda x: len([i for i in x.split(";") if i.strip()]))
for language in ["Python", "Java", "C++", "C", "JavaScript"]:
    col = "has_" + language.lower().replace("+","plus").replace("#","sharp")
    X[col] = X["languages"].str.contains(language, regex=False).astype(int)

X["certifications"] = X["certifications"].fillna("").astype(str)
X["certification_count"] = X["certifications"].apply(lambda x: len([i for i in x.split(";") if i.strip()]))
for cert in ["AI/ML", "Data Analytics", "Cloud Computing", "Cyber Security"]:
    col = "cert_" + cert.lower().replace("/","_").replace(" ","_")
    X[col] = X["certifications"].str.contains(cert, regex=False).astype(int)

X = X.drop(columns=["languages", "certifications"])
display(X.head())

## 8. Feature types and train/test split

In [ ]:
numeric_features = X.select_dtypes(include=["int64","float64","int32","float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object","category","bool"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 9. Preprocessing

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numerical", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 10. Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
logistic_model.fit(X_train, y_train)
lr_predictions = logistic_model.predict(X_test)

## 11. Decision Tree

In [ ]:
decision_tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini", max_depth=5, min_samples_split=10,
        min_samples_leaf=5, class_weight="balanced", random_state=42
    ))
])
decision_tree_model.fit(X_train, y_train)
dt_predictions = decision_tree_model.predict(X_test)

## 12. Evaluation

In [ ]:
def evaluate_model(name, y_true, y_pred):
    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }
    print("="*60)
    print(name)
    print("="*60)
    for k, v in result.items():
        if k != "Model":
            print(f"{k:10s}: {v:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=["Not Placed","Placed"], zero_division=0))
    return result

lr_result = evaluate_model("Logistic Regression", y_test, lr_predictions)
dt_result = evaluate_model("Decision Tree", y_test, dt_predictions)

## 13. Model comparison

In [ ]:
results_df = pd.DataFrame([lr_result, dt_result])
display(results_df.round(4))

results_df.set_index("Model")[["Precision","Recall","F1-Score"]].plot(
    kind="bar", figsize=(9,5), ylim=(0,1)
)
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

best_name = results_df.loc[results_df["F1-Score"].idxmax(), "Model"]
best_model = logistic_model if best_name == "Logistic Regression" else decision_tree_model
print("Best model by F1-score:", best_name)

## 14. Confusion matrices

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,5))
for axis, pred, title in [
    (ax[0], lr_predictions, "Logistic Regression"),
    (ax[1], dt_predictions, "Decision Tree")
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Not Placed","Placed"],
                yticklabels=["Not Placed","Placed"], ax=axis)
    axis.set_title(title)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 15. Decision Tree feature importance

In [ ]:
tree = decision_tree_model.named_steps["classifier"]
prep = decision_tree_model.named_steps["preprocessor"]
feature_names = prep.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance.head(20))

top = importance.head(15).sort_values("Importance")
plt.figure(figsize=(10,6))
plt.barh(top["Feature"], top["Importance"])
plt.title("Top Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 16. Decision Tree visualization

In [ ]:
plt.figure(figsize=(24,12))
plot_tree(tree, feature_names=feature_names,
          class_names=["Not Placed","Placed"],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title("Decision Tree — Top Levels")
plt.show()

## 17. Required example prediction

In [ ]:
example_index = X_test.index[0]
example = X_test.loc[[example_index]]
actual = int(y_test.loc[example_index])
pred = int(best_model.predict(example)[0])
prob = float(best_model.predict_proba(example)[0,1]) if hasattr(best_model, "predict_proba") else np.nan

actual_label = "Placed" if actual else "Not Placed"
pred_label = "Placed" if pred else "Not Placed"

print("Actual:", actual_label)
print("Predicted:", pred_label)
if not np.isnan(prob):
    print(f"Probability of Placed: {prob:.2%}")

if actual == pred:
    print("Explanation: The model correctly classified this held-out student.")
else:
    print("Explanation: The model misclassified this held-out student, showing that predictions are estimates and can contain errors.")

display(example)

## 18. Custom prediction

In [ ]:
custom_student = pd.DataFrame([{
    "gender":"Male", "age":21, "course":"B.Tech", "branch":"CSE", "year":4,
    "cgpa":8.0, "backlogs":0, "projects":3,
    "tech_skill":8, "comm_skill":7, "apt_skill":8, "team_skill":8,
    "participation":"Yes", "internship":"Yes", "training":"Yes",
    "language_count":3, "has_python":1, "has_java":1,
    "has_cplusplus":1, "has_c":0, "has_javascript":1,
    "certification_count":2, "cert_ai_ml":1,
    "cert_data_analytics":1, "cert_cloud_computing":0,
    "cert_cyber_security":0
}])

custom_pred = int(best_model.predict(custom_student)[0])
custom_prob = float(best_model.predict_proba(custom_student)[0,1])
print("Custom prediction:", "Placed" if custom_pred else "Not Placed")
print(f"Probability of Placed: {custom_prob:.2%}")

## 19. Save model

In [ ]:
MODEL_PATH = "placement_readiness_model.joblib"
joblib.dump(best_model, MODEL_PATH)
print("Saved:", MODEL_PATH)

## 20. Final checklist

The notebook covers the complete required pipeline: cleaning, duplicate handling, feature selection, feature engineering, missing-value treatment, categorical encoding, two-model comparison, Precision/Recall/F1 evaluation, example prediction and explanation, interpretability, and model export.